# CMSC 173 &middot; Module 9 &middot; Example 4
## Deep quakes, revisited: a curve instead of a line

Module 6 drew a straight line to find deep earthquakes (300 km or more) from where they struck. It found 73 of 82 test quakes, with 357 false alarms. Same data, same split: an RBF SVM draws a curve.

**How to use this notebook.** Runtime &rarr; Run all. Every number printed below is on the
slides; change something and see what moves.

In [ ]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

url = "https://raw.githubusercontent.com/njpinton/cmsc173-labs/main/data/ph_earthquakes.csv"
quakes = pd.read_csv(url, parse_dates=["time"])
quakes["deep"] = (quakes["depth"] >= 300).astype(int)
X, y = quakes[["longitude", "latitude"]], quakes["deep"]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=173, stratify=y)

svm = make_pipeline(StandardScaler(), SVC(class_weight="balanced")).fit(X_tr, y_tr)
print(round(svm.score(X_te, y_te), 3))           # 0.963
print(confusion_matrix(y_te, svm.predict(X_te)))  # all 82 deep quakes found; 128 false alarms

**The straight lines, for comparison.** Module 6's logistic regression, and an SVM with a linear kernel.

In [ ]:
for name, model in (("logistic regression", LogisticRegression(class_weight="balanced")),
                    ("linear SVM", SVC(kernel="linear", class_weight="balanced"))):
    m = make_pipeline(StandardScaler(), model).fit(X_tr, y_tr)
    print(name, round(m.score(X_te, y_te), 3), confusion_matrix(y_te, m.predict(X_te)).tolist())
# both lines find 73 of 82; the loss barely matters, the shape of the boundary does

**Try it.** Raise `gamma` to 10 inside `SVC(...)`. Do the false alarms fall? Does it still find every deep quake?